In [2]:
from sentence_transformers import SentenceTransformer


In [ ]:
# ==========================================
# CELDA 2 — RAG COMPLETO
# ==========================================

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
import os

# ==========================================
# CONFIG
# ==========================================

DOCUMENTS_PATH = r"C:\Users\UsuarioAdm\Desktop\documents"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

# ==========================================
# VERIFICAR DOCUMENTOS
# ==========================================

all_files = os.listdir(DOCUMENTS_PATH)

print("DOCUMENTOS ENCONTRADOS:\n")

print(all_files)

# ==========================================
# LEER PDF
# ==========================================

def read_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:

        extracted = page.extract_text()

        if extracted:
            text += extracted + "\n"

    return text


# ==========================================
# CHUNKING
# ==========================================

def chunk_text(
    text,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP
):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# ==========================================
# EMBEDDING MODEL
# ==========================================

print("\nCargando modelo embeddings...\n")

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("✅ Modelo cargado")


# ==========================================
# CHROMADB
# ==========================================

client = chromadb.PersistentClient(
    path="./chromadb"
)

collection = client.get_or_create_collection(
    name="fotomultas"
)

print("✅ ChromaDB listo")


# ==========================================
# INGESTA
# ==========================================

chunk_counter = 0

for file in all_files:

    if not file.endswith(".pdf"):
        continue

    print(f"\nProcesando: {file}")

    file_path = os.path.join(
        DOCUMENTS_PATH,
        file
    )

    text = read_pdf(file_path)

    print(f"Caracteres extraídos: {len(text)}")

    chunks = chunk_text(text)

    print(f"Chunks generados: {len(chunks)}")

    for idx, chunk in enumerate(chunks):

        embedding = model.encode(chunk)

        chunk_id = f"{file}_{idx}"

        collection.add(

            documents=[chunk],

            embeddings=[
                embedding.tolist()
            ],

            ids=[chunk_id],

            metadatas=[
                {
                    "source": file
                }
            ]
        )

        chunk_counter += 1


print("\n==============================")
print("✅ INGESTA TERMINADA")
print("==============================")

print(f"Total chunks: {chunk_counter}")


# ==========================================
# REGLAS LEGALES
# ==========================================

def apply_rules(user_input):

    rules_found = []

    text = user_input.lower()

    # --------------------------------------
    # NO CONDUCTOR
    # --------------------------------------

    if "yo no manejaba" in text:

        rules_found.append({

            "rule":
            "No conductor identificado",

            "probability":
            "Alta",

            "reason":
            "La autoridad debe probar quién conducía."

        })

    # --------------------------------------
    # NOTIFICACIÓN
    # --------------------------------------

    if "nunca me notificaron" in text:

        rules_found.append({

            "rule":
            "Posible falla de notificación",

            "probability":
            "Media-Alta",

            "reason":
            "Podría existir violación al debido proceso."

        })

    # --------------------------------------
    # TECNOMECÁNICA
    # --------------------------------------

    if "tecnomecanica" in text:

        if "estaba parqueado" in text:

            rules_found.append({

                "rule":
                "Vehículo no circulando",

                "probability":
                "Alta",

                "reason":
                "La infracción requiere circulación."

            })

    return rules_found


# ==========================================
# PIPELINE FINAL
# ==========================================

def analyze_case(user_query):

    print("\n=========================")
    print("ANÁLISIS DEL CASO")
    print("=========================\n")

    # --------------------------------------
    # EMBEDDING QUERY
    # --------------------------------------

    query_embedding = model.encode(user_query)

    # --------------------------------------
    # SEARCH
    # --------------------------------------

    results = collection.query(

        query_embeddings=[
            query_embedding.tolist()
        ],

        n_results=3
    )

    documents = results["documents"][0]

    metadatas = results["metadatas"][0]

    # --------------------------------------
    # RULES
    # --------------------------------------

    rules = apply_rules(user_query)

    print("CASO:\n")

    print(user_query)

    # --------------------------------------
    # REGLAS
    # --------------------------------------

    print("\n----------------------")
    print("REGLAS DETECTADAS")
    print("----------------------\n")

    if len(rules) == 0:

        print("No se detectaron reglas fuertes.")

    else:

        for rule in rules:

            print(f"Regla: {rule['rule']}")
            print(f"Probabilidad: {rule['probability']}")
            print(f"Razón: {rule['reason']}")
            print()

    # --------------------------------------
    # DOCUMENTOS
    # --------------------------------------

    print("\n----------------------")
    print("DOCUMENTOS RELEVANTES")
    print("----------------------\n")

    for doc, meta in zip(documents, metadatas):

        print("\n======================")
        print(f"FUENTE: {meta['source']}")
        print("======================\n")

        print(doc[:1000])


# ==========================================
# EJEMPLO
# ==========================================

case = """
Me llegó una fotomulta
de velocidad
pero yo no manejaba
y nunca me notificaron
"""

analyze_case(case)

DOCUMENTOS ENCONTRADOS:

['Ley_769_de_2002.pdf', 'Mal parqueo grua e inmovilizacion.pdf', 'Notificacion Multa.pdf', 'Proceso Fotodeteccion.pdf', 'SOAT accidentes y aseguradoras.pdf', 'SOAT tecnomecanica y velocidad.pdf', 'Tecnomecanica requiere vehiculo en circulacion.pdf', 'Velocidad y semaforos responsabilidad.pdf']

Cargando modelo embeddings...



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 575.37it/s]


✅ Modelo cargado


invalid pdf header: b'\t\n%PD'
EOF marker seems truncated
incorrect startxref pointer(1)
parsing for Object Streams


✅ ChromaDB listo

Procesando: Ley_769_de_2002.pdf
Caracteres extraídos: 228780
Chunks generados: 572

Procesando: Mal parqueo grua e inmovilizacion.pdf
Caracteres extraídos: 116493
Chunks generados: 292

Procesando: Notificacion Multa.pdf
Caracteres extraídos: 120407
Chunks generados: 302

Procesando: Proceso Fotodeteccion.pdf
Caracteres extraídos: 24596
Chunks generados: 62

Procesando: SOAT accidentes y aseguradoras.pdf
Caracteres extraídos: 60329
Chunks generados: 151

Procesando: SOAT tecnomecanica y velocidad.pdf
Caracteres extraídos: 328908
Chunks generados: 823

Procesando: Tecnomecanica requiere vehiculo en circulacion.pdf
Caracteres extraídos: 32714
Chunks generados: 82

Procesando: Velocidad y semaforos responsabilidad.pdf
Caracteres extraídos: 197130
Chunks generados: 493
